In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. GENERATE SYNTHETIC CREDIT CARD DATA
# (If using Google Cloud Storage, replace with: df = pd.read_csv('gs://your-bucket-name/data.csv'))
np.random.seed(42)
n_samples = 1000

data = {
    'Income': np.random.normal(50000, 15000, n_samples),
    'Age': np.random.randint(21, 65, n_samples),
    'Debt': np.random.normal(5000, 3000, n_samples),
    'Credit_Score': np.random.randint(300, 850, n_samples),
    'Prior_Default': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
}

df = pd.DataFrame(data)
# Rule for approval (1 = Approved, 0 = Denied)
df['Approved'] = ((df['Credit_Score'] > 600) & (df['Prior_Default'] == 0)).astype(int)

# 2. SEPARATE FEATURES AND TARGET
X = df.drop('Approved', axis=1)
y = df['Approved']

# 3. TRAIN-TEST SPLIT (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. INITIALIZE AND TRAIN THE RANDOM FOREST MODEL
# Note: Random Forest does NOT require scaling (StandardScaler), unlike Logistic Regression.
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 5. MAKE PREDICTIONS
y_pred = rf_model.predict(X_test)

# 6. EVALUATE THE MODEL
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

# 7. FEATURE IMPORTANCE (A great benefit of Random Forest)
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# --- OUTPUTS ---
print("==================================================")
print("           RANDOM FOREST MODEL OUTPUT             ")
print("==================================================")
print(f"Model Accuracy: {accuracy:.4f}\n")
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)
print("\nFeature Importances (Which features mattered most):")
print(feature_importance_df.to_string(index=False))

           RANDOM FOREST MODEL OUTPUT             
Model Accuracy: 0.9950

Confusion Matrix:
[[125   0]
 [  1  74]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       125
           1       1.00      0.99      0.99        75

    accuracy                           0.99       200
   macro avg       1.00      0.99      0.99       200
weighted avg       1.00      0.99      0.99       200


Feature Importances (Which features mattered most):
      Feature  Importance
 Credit_Score    0.750234
Prior_Default    0.204473
       Income    0.016168
         Debt    0.014872
          Age    0.014253
